# MeetingActionAgent: двухагентный помощник по протоколам совещаний

> На вход — текстовая стенограмма совещания; MinutesAgent извлекает протокол, ReviewAgent сверяет его с оригиналом.

Автор: [@Henry2513](https://github.com/Henry2513)

Дата: 2026-08-04

**Для кого**
- разработчики, впервые изучающие Agent или HelloAgents
- те, кто хочет понять схему «агент генерации + агент проверки»

**Предварительные требования**
- Python 3.11+
- установлен `requirements.txt`
- для реального запуска нужен OpenAI-compatible LLM API

**Цели обучения**
- последовательное взаимодействие двух `SimpleAgent`
- валидация JSON-ответа модели через Pydantic
- ограничение числа повторов и генерация результата в JSON и Markdown


## Маршрут обучения

1. Загрузка окружения и путей проекта
2. Определение структуры данных протокола
3. Разбор и рендеринг структурированного результата
4. Создание MinutesAgent и ReviewAgent
5. Оркестрация извлечения, проверки и одной правки
6. Запуск реального двухагентного процесса и самопроверка


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from hello_agents import HelloAgentsLLM, SimpleAgent
from pydantic import BaseModel, Field, ValidationError

PROJECT_ROOT = Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")
print(f"Каталог проекта: {PROJECT_ROOT}")



## 1. Архитектура двух агентов

```text
Текстовая стенограмма совещания
  → MinutesAgent: извлекает только информацию, подтверждённую оригиналом
  → ReviewAgent: проверяет пропуски, выдумки и противоречия
  → при необходимости — одна правка и повторная проверка
  → JSON + Markdown
```

В первой версии вызов инструментов не используется. Обычный Python отвечает за чтение, валидацию и сохранение файлов; агенты занимаются только языковым пониманием и проверкой.


In [ ]:
# Одно действие (action item), извлечённое из оригинала совещания.
class ActionItem(BaseModel):
    task: str = Field(min_length=1)
    owner: str | None = None
    due_date_raw: str | None = None
    priority: Literal["高", "中", "低", "未说明"] = "未说明"
    evidence: str = Field(min_length=1)


# Полный протокол совещания и статус проверки.
class MeetingResult(BaseModel):
    title: str = Field(min_length=1)
    meeting_date: str | None = None
    participants: list[str] = Field(default_factory=list)
    summary: str = Field(min_length=1)
    decisions: list[str] = Field(default_factory=list)
    action_items: list[ActionItem] = Field(default_factory=list)
    open_questions: list[str] = Field(default_factory=list)
    review_status: Literal["pending", "passed", "needs_manual_review"] = "pending"
    review_issues: list[str] = Field(default_factory=list)


# Заключение ReviewAgent по черновику протокола.
class ReviewResult(BaseModel):
    passed: bool
    issues: list[str] = Field(default_factory=list)
    missing_items: list[str] = Field(default_factory=list)
    unsupported_items: list[str] = Field(default_factory=list)
    revision_advice: list[str] = Field(default_factory=list)




## 2. Разбор JSON из ответа агента

Модель иногда оборачивает JSON в Markdown-ограждения или добавляет пояснение до/после. Функция ниже извлекает внешний JSON-объект и передаёт его в Pydantic для валидации.


In [ ]:
# Извлечь и разобрать JSON-объект из ответа модели.
def extract_json_object(text: str) -> dict:
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end < start:
        raise ValueError("В ответе модели нет полного JSON-объекта")
    return json.loads(text[start : end + 1])


# Разобрать ответ модели и провалидировать как указанную Pydantic-модель.
def parse_model_response(text: str, model_type: type[BaseModel]) -> BaseModel:
    return model_type.model_validate(extract_json_object(text))



## 3. Преобразование структурированного результата в Markdown

Markdown генерируется обычным Python, чтобы модель не переписывала уже проверенное содержимое.


In [ ]:
# Подготовить nullable-текст для Markdown-таблицы.
def markdown_cell(value: str | None) -> str:
    if not value:
        return "не указано"
    return value.replace("|", "\|").replace("\n", " ")


# Преобразовать протокол совещания в Markdown.
def to_markdown(result: MeetingResult) -> str:
    meeting_date = markdown_cell(result.meeting_date)
    participants = "、".join(result.participants) if result.participants else "не указано"
    lines = [
        f"# {result.title}",
        "",
        f"**Дата совещания:** {meeting_date}",
        "",
        f"**Участники:** {participants}",
        "",
        "## Краткое содержание",
        "",
        result.summary,
        "",
        "## Подтверждённые решения",
        "",
    ]
    lines.extend([f"- {item}" for item in result.decisions] or ["- нет"])
    lines.extend([
        "",
        "## Действия",
        "",
        "| Задача | Ответственный | Срок (из оригинала) | Приоритет | Цитата из оригинала |",
        "|---|---|---|---|---|",
    ])
    if result.action_items:
        for item in result.action_items:
            lines.append(
                "| "
                + " | ".join([
                    markdown_cell(item.task),
                    markdown_cell(item.owner),
                    markdown_cell(item.due_date_raw),
                    markdown_cell(item.priority),
                    markdown_cell(item.evidence),
                ])
                + " |"
            )
    else:
        lines.append("| нет | не указано | не указано | не указано | не указано |")

    lines.extend(["", "## Открытые вопросы", ""])
    lines.extend([f"- {item}" for item in result.open_questions] or ["- нет"])
    lines.extend(["", "## Статус проверки", "", f"`{result.review_status}`"])
    if result.review_issues:
        lines.extend(["", "### Замечания проверки", ""])
        lines.extend([f"- {item}" for item in result.review_issues])
    return "\n".join(lines) + "\n"


# Сохранить результат совещания в JSON и Markdown.
def save_result(result: MeetingResult, stem: str = "meeting_result") -> tuple[Path, Path]:
    output_dir = PROJECT_ROOT / "outputs"
    json_path = output_dir / f"{stem}.json"
    markdown_path = output_dir / f"{stem}.md"
    json_path.write_text(result.model_dump_json(indent=2), encoding="utf-8")
    markdown_path.write_text(to_markdown(result), encoding="utf-8")
    return json_path, markdown_path



## 4. Роли агентов и промпты

- MinutesAgent может извлекать только информацию, подтверждённую оригиналом; неизвестные поля остаются пустыми.
- ReviewAgent обязан сверять оригинал и черновик, проверяя пропуски, выдумки, конфликты дат и ошибочную запись «предложений» как «решений».


In [ ]:
MINUTES_SYSTEM_PROMPT = """Вы — строгий эксперт по извлечению протоколов совещаний на русском языке.
Используйте только информацию, явно подтверждённую текстом совещания; не дополняйте здравым смыслом и не угадывайте.
Чётко различайте обсуждение, предложения и подтверждённые решения.
Для каждого действия сохраняйте цитату из оригинала; неизвестные ответственный, дата совещания или срок — null.
Возвращайте только JSON по заданной схеме, без Markdown и лишних пояснений."""

REVIEW_SYSTEM_PROMPT = """Вы — независимый рецензент протоколов совещаний.
Сверяйте оригинал и черновик по пунктам: пропуски, выдумки, размытые действия и конфликты дат.
Не одобряйте только из-за гладкого текста и не используйте внешнюю информацию.
Возвращайте только JSON по заданной схеме, без Markdown и лишних пояснений."""



## 5. Ограниченное число вызовов и исправление формата

Весь процесс использует бюджет из четырёх вызовов модели. При первой ошибке разбора JSON допускается один запрос на исправление формата, он тоже учитывается в бюджете.


In [ ]:
class CallBudget:
    # Инициализация лимита вызовов модели.
    def __init__(self, maximum: int = 4) -> None:
        self.maximum = maximum
        self.used = 0

    # Оставшееся число вызовов модели.
    @property
    def remaining(self) -> int:
        return self.maximum - self.used

    # Выполнить один вызов агента в пределах лимита.
    def run(self, agent, prompt: str) -> str:
        if self.remaining <= 0:
            raise RuntimeError("Достигнут лимит в четыре вызова модели")
        self.used += 1
        return agent.run(prompt)


# Вызвать агента и разобрать ответ в указанную модель данных.
def run_structured(
    agent,
    prompt: str,
    model_type: type[BaseModel],
    budget: CallBudget,
) -> BaseModel:
    schema = json.dumps(model_type.model_json_schema(), ensure_ascii=False)
    full_prompt = f"{prompt}\n\nНеобходимо следовать следующей JSON Schema:\n{schema}"
    raw_response = budget.run(agent, full_prompt)
    try:
        return parse_model_response(raw_response, model_type)
    except ValueError as error:
        if budget.remaining <= 0:
            raise RuntimeError(f"Ошибка валидации JSON и нет оставшихся вызовов: {error}") from error
        repair_prompt = (
            "Предыдущий ответ не прошёл JSON-валидацию. Не меняйте смысл, исправьте только формат.\n"
            f"Ошибка валидации: {error}\n"
            f"Исходный ответ:\n{raw_response}\n"
            f"Целевая Schema:\n{schema}\n"
            "Верните только исправленный JSON."
        )
        repaired_response = budget.run(agent, repair_prompt)
        return parse_model_response(repaired_response, model_type)


# Создать MinutesAgent и ReviewAgent.
def build_agents():
    llm = HelloAgentsLLM()
    minutes_agent = SimpleAgent(name="MinutesAgent", llm=llm, system_prompt=MINUTES_SYSTEM_PROMPT)
    review_agent = SimpleAgent(name="ReviewAgent", llm=llm, system_prompt=REVIEW_SYSTEM_PROMPT)
    return minutes_agent, review_agent



## 6. Полный процесс анализа

При успешной первой проверке — два вызова модели; при неудаче и наличии двух оставшихся вызовов MinutesAgent исправляет один раз, затем ReviewAgent делает финальную проверку.


In [ ]:
# Проверить и очистить входной текст совещания.
def validate_transcript(transcript: str) -> str:
    cleaned = transcript.strip()
    if len(cleaned) < 20:
        raise ValueError("Стенограмма слишком короткая — укажите минимум 20 символов")
    return cleaned


# Сформировать промпт для проверки черновика протокола.
def make_review_prompt(transcript: str, draft: MeetingResult) -> str:
    return (
        "Проверьте следующий черновик протокола совещания.\n\n"
        f"【Оригинал совещания】\n{transcript}\n\n"
        f"【Черновик протокола】\n{draft.model_dump_json(indent=2)}"
    )


# Полный процесс: извлечение, проверка и при необходимости одна правка.
def analyze_meeting(transcript: str) -> tuple[MeetingResult, ReviewResult, int]:
    transcript = validate_transcript(transcript)
    minutes_agent, review_agent = build_agents()
    budget = CallBudget(maximum=4)

    draft_prompt = f"Извлеките структурированный протокол из следующего текста совещания:\n\n{transcript}"
    draft = run_structured(minutes_agent, draft_prompt, MeetingResult, budget)
    review = run_structured(review_agent, make_review_prompt(transcript, draft), ReviewResult, budget)

    if review.passed:
        final_result = draft.model_copy(update={"review_status": "passed", "review_issues": []})
        return final_result, review, budget.used

    issues = review.issues + review.missing_items + review.unsupported_items
    if budget.remaining < 2:
        final_result = draft.model_copy(
            update={"review_status": "needs_manual_review", "review_issues": issues}
        )
        return final_result, review, budget.used

    revision_prompt = (
        "Исправьте протокол по замечаниям проверки. По-прежнему используйте только информацию из оригинала.\n\n"
        f"【Оригинал совещания】\n{transcript}\n\n"
        f"【Исходный черновик】\n{draft.model_dump_json(indent=2)}\n\n"
        f"【Замечания проверки】\n{review.model_dump_json(indent=2)}"
    )
    revised = run_structured(minutes_agent, revision_prompt, MeetingResult, budget)
    final_review = run_structured(review_agent, make_review_prompt(transcript, revised), ReviewResult, budget)
    final_issues = final_review.issues + final_review.missing_items + final_review.unsupported_items
    status = "passed" if final_review.passed else "needs_manual_review"
    final_result = revised.model_copy(update={"review_status": status, "review_issues": final_issues})
    return final_result, final_review, budget.used



## 7. Запуск примера с двумя агентами

Эта ячейка читает конфигурацию модели и запускает MinutesAgent и ReviewAgent. Перед запуском укажите в `.env` значения `LLM_MODEL_ID`, `LLM_API_KEY` и `LLM_BASE_URL`.


In [ ]:
sample_transcript = (PROJECT_ROOT / "data" / "sample_meeting.txt").read_text(encoding="utf-8")
result, _, call_count = analyze_meeting(sample_transcript)
json_path, markdown_path = save_result(result)
print(f"Число вызовов модели: {call_count}")
print(f"Статус проверки: {result.review_status}")
print(f"Сохранено: {json_path.name}, {markdown_path.name}")



## 8. Самопроверка структуры и оркестрации

Эти проверки не вызывают модель; они проверяют Pydantic-структуры, извлечение JSON, отсутствующие поля, пустые действия, пустой ввод и рендеринг Markdown.


In [ ]:
expected_path = PROJECT_ROOT / "outputs" / "example_result.json"
expected_result = MeetingResult.model_validate_json(expected_path.read_text(encoding="utf-8"))

fenced = "```json\n" + expected_result.model_dump_json() + "\n```"
assert parse_model_response(fenced, MeetingResult).title == "Итерация функции регистрации нового пользователя"
assert expected_result.meeting_date == "2026-07-27"
assert expected_result.action_items[-1].owner is None

empty_actions = MeetingResult(
    title="Информационная синхронизация",
    participants=[],
    summary="На совещании только синхронизировали информацию, действий не сформировано.",
    decisions=[],
    action_items=[],
    open_questions=[],
)
assert empty_actions.action_items == []
assert "| нет |" in to_markdown(empty_actions)

try:
    validate_transcript("коротко")
except ValueError:
    pass
else:
    raise AssertionError("Слишком короткая стенограмма должна отклоняться")

try:
    ActionItem(task="тест", owner=None, due_date_raw=None, priority="紧急", evidence="оригинал")
except ValidationError:
    pass
else:
    raise AssertionError("Недопустимый приоритет должен отклоняться Pydantic")

rendered = to_markdown(expected_result)
tracked_markdown = (PROJECT_ROOT / "outputs" / "example_minutes.md").read_text(encoding="utf-8")
assert rendered == tracked_markdown
print("Самопроверка пройдена: 6 групп")



## Упражнение: анализ пограничного совещания

Откройте `data/edge_case_meeting.txt` и сначала предскажите результат вручную:

1. Является ли «можно рассмотреть запуск в следующем месяце» подтверждённым решением?
2. Есть ли ответственный за подтверждение тестовой среды?
3. Есть ли конфликт дат в продуктовой документации?

После включения реальной модели передайте `edge_transcript` в `analyze_meeting` и сравните с прогнозом.


In [ ]:
edge_transcript = (PROJECT_ROOT / "data" / "edge_case_meeting.txt").read_text(encoding="utf-8")
answer_scaffold = {
    "confirmed_launch_decision": False,
    "test_environment_owner": None,
    "document_date_conflict": True,
}
answer_scaffold



## Частые вопросы и следующие шаги

- **Предложения записаны как решения**: Reviewer должен проверять формулировки вроде «рассмотреть», «предложить», «возможно».
- **Выдуманные ответственные или даты**: неизвестные значения — `null`, в Markdown отображается «не указано».
- **Нестабильный JSON**: допускается одно исправление формата, но лимит — четыре вызова.
- **Несколько совещаний подряд**: каждый вызов `analyze_meeting` создаёт новых агентов, чтобы история не смешивалась.

Во второй версии можно добавить инструмент нормализации дат; в первой версии — два агента без вызова инструментов.
